# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [2]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始
CANN包路径 = /home/developer/Ascend/cann
教程目录 = /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


In [3]:
import importlib
import os
import site
import sys
from pathlib import Path

# 必须在 import torch 之前设置
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"

cann_root = Path(os.environ["ASCEND_TOOLKIT_HOME"]).resolve()

cann_python_paths = [
    cann_root / "python" / "site-packages",
    cann_root / "opp" / "built-in" / "op_impl" / "ai_core" / "tbe",
]

for path in cann_python_paths:
    if not path.exists():
        raise FileNotFoundError(f"CANN Python 路径不存在: {path}")

    path_str = str(path)

    # 处理目录中的 .pth 文件
    site.addsitedir(path_str)

    # 放到当前 Python 解释器的模块搜索路径前面
    if path_str in sys.path:
        sys.path.remove(path_str)
    sys.path.insert(0, path_str)

# 同步 PYTHONPATH，供 GE 内嵌 Python 使用
old_parts = [
    p for p in os.environ.get("PYTHONPATH", "").split(":") if p
]

new_parts = [str(path) for path in cann_python_paths]

for path in old_parts:
    if path not in new_parts:
        new_parts.append(path)

os.environ["PYTHONPATH"] = ":".join(new_parts)
importlib.invalidate_caches()

print("CANN root =", cann_root)
print("PYTHONPATH =", os.environ["PYTHONPATH"])

print("\nsys.path 前几项：")
for i, path in enumerate(sys.path[:8]):
    print(i, path)

# 在导入 torch 前验证 TBE
import tbe
import tbe.common

print("\nTBE 导入成功")
print("TBE 路径 =", tbe.__file__)

CANN root = /home/developer/Ascend/cann-9.0.0
PYTHONPATH = /home/developer/Ascend/cann-9.0.0/python/site-packages:/home/developer/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe

sys.path 前几项：
0 /home/developer/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe
1 /home/developer/Ascend/cann-9.0.0/python/site-packages
2 /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src
3 /opt/buildtools/Python-3.11.4/lib/python311.zip
4 /opt/buildtools/Python-3.11.4/lib/python3.11
5 /opt/buildtools/Python-3.11.4/lib/python3.11/lib-dynload
6 
7 /home/developer/.local/lib/python3.11/site-packages

TBE 导入成功
TBE 路径 = /home/developer/Ascend/cann-9.0.0/python/site-packages/tbe/__init__.py


In [4]:
import torch
import torch_npu

print("torch version     =", torch.__version__)
print("torch path        =", torch.__file__)
print("torch_npu version =", torch_npu.__version__)
print("torch_npu path    =", torch_npu.__file__)
print("NPU device        =", torch.npu.get_device_name(0))

assert torch.__version__.startswith("2.8.0"), torch.__version__
assert torch_npu.__version__ == "2.8.0.post4", torch_npu.__version__

print("TorchNPU 导入检查 PASS")

torch version     = 2.8.0+cpu
torch path        = /home/developer/.local/lib/python3.11/site-packages/torch/__init__.py
torch_npu version = 2.8.0.post4
torch_npu path    = /home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py
NPU device        = Ascend910B3
TorchNPU 导入检查 PASS


/home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


In [5]:
a = torch.randn(
    16,
    16,
    dtype=torch.float16,
    device="npu",
)

b = torch.randn(
    16,
    16,
    dtype=torch.float16,
    device="npu",
)

c = torch.matmul(a, b)
torch.npu.synchronize()

print("普通 NPU Matmul PASS")
print("shape  =", c.shape)
print("dtype  =", c.dtype)
print("device =", c.device)

普通 NPU Matmul PASS
shape  = torch.Size([16, 16])
dtype  = torch.float16
device = npu:0


In [6]:
torch.npu.config.allow_internal_format = True

x = torch.randint(
    -128,
    127,
    (32, 64),
    dtype=torch.int8,
    device="npu",
)

print("转换前 format =", torch_npu.get_npu_format(x))

x_nz = torch_npu.npu_format_cast(x, 29)
torch.npu.synchronize()

print("转换后 format =", torch_npu.get_npu_format(x_nz))
print("INT8 ND -> FRACTAL_NZ PASS")

转换前 format = ND
转换后 format = FRACTAL_NZ
INT8 ND -> FRACTAL_NZ PASS


In [1]:
import os
print("当前 Notebook 内核 PID =", os.getpid())

当前 Notebook 内核 PID = 8615


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 MB 41.2 MB/s  0:00:02:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.4/33.4 MB 54.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torch_npu]/2 [torch_npu]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.22.1 requires torch==2.7.1, but you have torch 2.8.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [7]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
// qmm_custom.asc
// Kernel direct-call implementation for A8W8 quantized matmul on Ascend 910B.

#include <cstdint>
#include <limits>

#include <torch/extension.h>
#include "torch_npu/csrc/core/npu/NPUStream.h"

#include "kernel_tiling/kernel_tiling.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"

using namespace AscendC;
using namespace matmul;

struct QmmCustomTilingData {
    AscendC::tiling::TCubeTiling cubeTilingData;
    uint32_t isPertoken;
    uint32_t reserved;
    uint64_t workspaceSize;
};

static QmmCustomTilingData CalcQmmTiling(
    uint32_t isPertoken, int64_t m64, int64_t n64, int64_t k64)
{
    TORCH_CHECK(m64 > 0 && n64 > 0 && k64 > 0,
                "qmm_custom expects positive M, N and K, but got M=", m64,
                ", N=", n64, ", K=", k64);
    TORCH_CHECK(m64 <= std::numeric_limits<int32_t>::max() &&
                    n64 <= std::numeric_limits<int32_t>::max() &&
                    k64 <= std::numeric_limits<int32_t>::max(),
                "qmm_custom only supports dimensions representable by int32");

    const int32_t m = static_cast<int32_t>(m64);
    const int32_t n = static_cast<int32_t>(n64);
    const int32_t k = static_cast<int32_t>(k64);

    auto *platform = platform_ascendc::PlatformAscendCManager::GetInstance();
    TORCH_CHECK(platform != nullptr,
                "PlatformAscendCManager::GetInstance() returned nullptr");

    const uint32_t aicNum = platform->GetCoreNumAic();
    const uint32_t aivNum = platform->GetCoreNumAiv();
    TORCH_CHECK(aicNum > 0 && aivNum > 0,
                "No available AIC/AIV cores: aic=", aicNum,
                ", aiv=", aivNum);

    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*platform);

    // Preserve the proven single-operator policy.  The narrow-M/N=6144 case
    // stays on one Cube core to avoid an ambiguous physical-NZ submatrix
    // offset; all other shapes are distributed over the available Cube cores.
    const uint32_t matmulDim =
        (m < 128 && n == 6144) ? 1U : aicNum;
    cubeTiling.SetDim(matmulDim);
    cubeTiling.SetAType(matmul_tiling::TPosition::GM,
                        matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_INT8);
    cubeTiling.SetBType(matmul_tiling::TPosition::GM,
                        matmul_tiling::CubeFormat::NZ,
                        matmul_tiling::DataType::DT_INT8);
    cubeTiling.SetCType(matmul_tiling::TPosition::GM,
                        matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_INT32);
    cubeTiling.SetBiasType(matmul_tiling::TPosition::GM,
                           matmul_tiling::CubeFormat::ND,
                           matmul_tiling::DataType::DT_INT32);

    // Vector dequantization needs a relatively small base tile to keep the
    // INT32/F32/BF16 temporary tensors inside UB.  Cube-only can use a larger M
    // tile because it has no Vector-side result tile.
    if (isPertoken != 0U) {
        cubeTiling.SetFixSplit(m < 32 ? 16 : 32, 64, -1);
    } else {
        cubeTiling.SetFixSplit(m < 128 ? 16 : 128, 64, -1);
    }

    cubeTiling.SetShape(m, n, k);
    cubeTiling.SetOrgShape(m, n, k);
    cubeTiling.SetBias(false);
    cubeTiling.SetBufferSpace(-1, -1, -1);

    QmmCustomTilingData result{};
    result.isPertoken = isPertoken;
    result.reserved = 0U;
    const int32_t tilingStatus = cubeTiling.GetTiling(result.cubeTilingData);
    TORCH_CHECK(tilingStatus != -1,
                "GetMatmulTiling failed for M=", m,
                ", N=", n, ", K=", k);
    TORCH_CHECK(result.cubeTilingData.usedCoreNum > 0,
                "GetMatmulTiling returned usedCoreNum=0");

    const uint64_t systemWorkspaceSize =
        static_cast<uint64_t>(platform->GetLibApiWorkSpaceSize());
    const uint64_t userWorkspaceSize =
        isPertoken == 0U
            ? 0ULL
            : static_cast<uint64_t>(m) * static_cast<uint64_t>(n) *
                  sizeof(int32_t);
    TORCH_CHECK(userWorkspaceSize <=
                    std::numeric_limits<uint64_t>::max() - systemWorkspaceSize,
                "workspace size overflow");

    result.workspaceSize = systemWorkspaceSize + userWorkspaceSize;
    // This entry is MIX(1,2) even for Path 1.  Keep a non-zero allocation in
    // the unlikely case that the reported Matmul system workspace is zero.
    if (result.workspaceSize == 0U) {
        result.workspaceSize = 32U;
    }
    return result;
}

__aicore__ inline uint32_t QmmCeilDiv(uint32_t value, uint32_t divisor)
{
    return (value + divisor - 1U) / divisor;
}

__aicore__ inline uint32_t MinU32(uint32_t lhs, uint32_t rhs)
{
    return lhs < rhs ? lhs : rhs;
}

// Use a flag ID outside the Matmul client's internal event range.
constexpr uint64_t QMM_SYNC_AIC_AIV_FLAG = 8;
constexpr uint64_t QMM_SYNC_MODE2 = 2;
constexpr MatmulConfig QMM_NZ_CFG_MDL =
    GetMDLConfig(false, false, 0, true, false, false, false);

template <typename InputType>
class QmmCubeBasicKernel {
public:
    using AType = MatmulType<TPosition::GM, CubeFormat::ND, InputType>;
    using BType = MatmulType<TPosition::GM, CubeFormat::NZ, InputType>;
    using CType = MatmulType<TPosition::GM, CubeFormat::ND, int32_t>;
    using BiasType = MatmulType<TPosition::GM, CubeFormat::ND, int32_t>;

    MatmulImpl<AType, BType, CType, BiasType, QMM_NZ_CFG_MDL> matmulObj;

    __aicore__ inline void Init(
        GM_ADDR x1, GM_ADDR x2, GM_ADDR y, const TCubeTiling &tiling)
    {
        tiling_ = tiling;
        coreIndex_ = GetBlockIdx();
        const uint32_t taskRatio = GetTaskRation();
        if (taskRatio > 1U) {
            coreIndex_ /= taskRatio;
        }
        active_ = coreIndex_ < tiling_.usedCoreNum;
        if (!active_) {
            return;
        }

        const uint32_t mBlockCount =
            QmmCeilDiv(tiling_.M, tiling_.singleCoreM);
        mIndex_ = coreIndex_ % mBlockCount;
        nIndex_ = coreIndex_ / mBlockCount;
        mUse_ = MinU32(
            tiling_.singleCoreM, tiling_.M - mIndex_ * tiling_.singleCoreM);
        nUse_ = MinU32(
            tiling_.singleCoreN, tiling_.N - nIndex_ * tiling_.singleCoreN);

        x1Global_.SetGlobalBuffer(
            reinterpret_cast<__gm__ InputType *>(x1),
            static_cast<uint64_t>(tiling_.M) * tiling_.Ka);
        x2Global_.SetGlobalBuffer(
            reinterpret_cast<__gm__ InputType *>(x2),
            static_cast<uint64_t>(tiling_.Kb) * tiling_.N);
        yGlobal_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(y),
            static_cast<uint64_t>(tiling_.M) * tiling_.N);

        const uint64_t x1Offset =
            static_cast<uint64_t>(mIndex_) * tiling_.singleCoreM * tiling_.Ka;
        // B is FRACTAL_NZ. A complete N tile occupies singleCoreN * K
        // elements, unlike the ND offset used by the basic sample.
        const uint64_t x2Offset =
            static_cast<uint64_t>(nIndex_) * tiling_.singleCoreN * tiling_.Kb;
        const uint64_t yOffset =
            static_cast<uint64_t>(mIndex_) * tiling_.singleCoreM * tiling_.N +
            static_cast<uint64_t>(nIndex_) * tiling_.singleCoreN;

        x1Global_ = x1Global_[x1Offset];
        x2Global_ = x2Global_[x2Offset];
        yGlobal_ = yGlobal_[yOffset];
        matmulObj.SetOrgShape(tiling_.M, tiling_.N, tiling_.Ka);
    }

    __aicore__ inline void Process()
    {
        if (!active_ || GetSysWorkSpacePtr() == nullptr) {
            return;
        }
        if ASCEND_IS_AIC {
            matmulObj.SetOrgShape(
                tiling_.M, tiling_.N, tiling_.Ka);
            matmulObj.SetSingleShape(mUse_, nUse_, tiling_.Ka);
            matmulObj.SetTensorA(x1Global_);
            matmulObj.SetTensorB(x2Global_);
            while (matmulObj.Iterate()) {
                matmulObj.GetTensorC(yGlobal_, 0, false);
            }
        }
    }

private:
    GlobalTensor<InputType> x1Global_;
    GlobalTensor<InputType> x2Global_;
    GlobalTensor<int32_t> yGlobal_;
    TCubeTiling tiling_;
    uint32_t coreIndex_ = 0;
    uint32_t mIndex_ = 0;
    uint32_t nIndex_ = 0;
    uint32_t mUse_ = 0;
    uint32_t nUse_ = 0;
    bool active_ = false;
};

template <typename InputType>
class QmmPertokenKernel {
public:
    using AType = MatmulType<TPosition::GM, CubeFormat::ND, InputType>;
    using BType = MatmulType<TPosition::GM, CubeFormat::NZ, InputType>;
    using CType = MatmulType<TPosition::GM, CubeFormat::ND, int32_t>;
    using BiasType = MatmulType<TPosition::GM, CubeFormat::ND, int32_t>;

    MatmulImpl<AType, BType, CType, BiasType, QMM_NZ_CFG_MDL> matmulObj;

    __aicore__ inline void Init(
        GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale,
        GM_ADDR y, GM_ADDR userWorkspace, const TCubeTiling &tiling,
        TPipe *pipe)
    {
        tiling_ = tiling;
        coreIndex_ = GetBlockIdx();
        const uint32_t taskRatio = GetTaskRation();
        if (taskRatio > 1U) {
            coreIndex_ /= taskRatio;
        }
        active_ = coreIndex_ < tiling_.usedCoreNum;
        if (!active_) {
            return;
        }

        const uint32_t mBlockCount =
            QmmCeilDiv(tiling_.M, tiling_.singleCoreM);
        mIndex_ = coreIndex_ % mBlockCount;
        nIndex_ = coreIndex_ / mBlockCount;
        mUse_ = MinU32(
            tiling_.singleCoreM, tiling_.M - mIndex_ * tiling_.singleCoreM);
        nUse_ = MinU32(
            tiling_.singleCoreN, tiling_.N - nIndex_ * tiling_.singleCoreN);

        x1Global_.SetGlobalBuffer(
            reinterpret_cast<__gm__ InputType *>(x1),
            static_cast<uint64_t>(tiling_.M) * tiling_.Ka);
        x2Global_.SetGlobalBuffer(
            reinterpret_cast<__gm__ InputType *>(x2),
            static_cast<uint64_t>(tiling_.Kb) * tiling_.N);
        scaleGlobal_.SetGlobalBuffer(
            reinterpret_cast<__gm__ float *>(scale), tiling_.N);
        pertokenGlobal_.SetGlobalBuffer(
            reinterpret_cast<__gm__ float *>(pertokenScale), tiling_.M);
        yGlobal_.SetGlobalBuffer(
            reinterpret_cast<__gm__ bfloat16_t *>(y),
            static_cast<uint64_t>(tiling_.M) * tiling_.N);
        mmWorkspaceGlobal_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(userWorkspace),
            static_cast<uint64_t>(tiling_.M) * tiling_.N);

        const uint64_t x1Offset =
            static_cast<uint64_t>(mIndex_) * tiling_.singleCoreM * tiling_.Ka;
        const uint64_t x2Offset =
            static_cast<uint64_t>(nIndex_) * tiling_.singleCoreN * tiling_.Kb;
        const uint64_t yOffset =
            static_cast<uint64_t>(mIndex_) * tiling_.singleCoreM * tiling_.N +
            static_cast<uint64_t>(nIndex_) * tiling_.singleCoreN;

        x1Global_ = x1Global_[x1Offset];
        x2Global_ = x2Global_[x2Offset];
        scaleGlobal_ = scaleGlobal_[
            static_cast<uint64_t>(nIndex_) * tiling_.singleCoreN];
        pertokenGlobal_ = pertokenGlobal_[
            static_cast<uint64_t>(mIndex_) * tiling_.singleCoreM];
        yGlobal_ = yGlobal_[yOffset];
        mmWorkspaceGlobal_ = mmWorkspaceGlobal_[yOffset];

        matmulObj.SetOrgShape(tiling_.M, tiling_.N, tiling_.Ka);

        tileElements_ = tiling_.baseM * tiling_.baseN;
        if ASCEND_IS_AIV {
            pipe->InitBuffer(
                mmOutQueue_, 1, tileElements_ * sizeof(int32_t));
            pipe->InitBuffer(
                scaleQueue_, 1, tiling_.baseN * sizeof(float));
            pipe->InitBuffer(
                pertokenQueue_, 1, tiling_.baseM * sizeof(float));
            pipe->InitBuffer(
                outQueue_, 1, tileElements_ * sizeof(bfloat16_t));
            // float result + per-token broadcast + mul result + two
            // result-sized regions shared by Dequant and BroadCast.
            pipe->InitBuffer(
                workBuffer_, 5U * tileElements_ * sizeof(float));

            dequantResult_ = workBuffer_.GetWithOffset<float>(
                tileElements_, 0);
            pertokenBroadcast_ = workBuffer_.GetWithOffset<float>(
                tileElements_, tileElements_ * sizeof(float));
            mulResult_ = workBuffer_.GetWithOffset<float>(
                tileElements_, 2U * tileElements_ * sizeof(float));
            sharedTmp_ = workBuffer_.GetWithOffset<uint8_t>(
                2U * tileElements_ * sizeof(float),
                3U * tileElements_ * sizeof(float));
        }
    }

    __aicore__ inline void Process()
    {
        if (!active_ || GetSysWorkSpacePtr() == nullptr) {
            return;
        }

        if ASCEND_IS_AIC {
            matmulObj.SetOrgShape(
                tiling_.M, tiling_.N, tiling_.Ka);
            matmulObj.SetSingleShape(mUse_, nUse_, tiling_.Ka);
            matmulObj.SetTensorA(x1Global_);
            matmulObj.SetTensorB(x2Global_);
            while (matmulObj.Iterate()) {
                matmulObj.GetTensorC(
                    mmWorkspaceGlobal_, 0, false);
            }
            CrossCoreSetFlag<QMM_SYNC_MODE2, PIPE_FIX>(
                QMM_SYNC_AIC_AIV_FLAG);
        }

        if ASCEND_IS_AIV {
            CrossCoreWaitFlag(QMM_SYNC_AIC_AIV_FLAG);
            const uint32_t roundM =
                QmmCeilDiv(mUse_, tiling_.baseM);
            const uint32_t roundN =
                QmmCeilDiv(nUse_, tiling_.baseN);
            const uint32_t taskRatio = GetTaskRation();
            const uint32_t subBlockIndex = GetSubBlockIdx();
            const uint32_t totalRounds = roundM * roundN;
            for (uint32_t computeRound = subBlockIndex;
                 computeRound < totalRounds;
                 computeRound += taskRatio) {
                const uint32_t tileMIndex = computeRound % roundM;
                const uint32_t tileNIndex = computeRound / roundM;
                const uint32_t mOffset =
                    tileMIndex * tiling_.baseM;
                const uint32_t nOffset =
                    tileNIndex * tiling_.baseN;
                const uint32_t currentM =
                    MinU32(tiling_.baseM, mUse_ - mOffset);
                const uint32_t currentN =
                    MinU32(tiling_.baseN, nUse_ - nOffset);
                ProcessTile(
                    currentM, currentN, mOffset, nOffset);
            }
        }
    }

private:
    __aicore__ inline LocalTensor<float> CopyScale(
        uint32_t currentN, uint32_t nOffset)
    {
        LocalTensor<float> scaleLocal = scaleQueue_.AllocTensor<float>();
        DataCopy(scaleLocal, scaleGlobal_[nOffset], currentN);
        scaleQueue_.EnQue(scaleLocal);
        scaleLocal = scaleQueue_.DeQue<float>();
        scaleLocal.SetSize(tiling_.baseN);
        return scaleLocal;
    }

    __aicore__ inline LocalTensor<float> CopyPertoken(
        uint32_t currentM, uint32_t mOffset)
    {
        LocalTensor<float> pertokenLocal =
            pertokenQueue_.AllocTensor<float>();
        DataCopyExtParams params;
        params.blockCount = 1;
        params.blockLen = currentM * sizeof(float);
        params.srcStride = 0;
        params.dstStride = 0;
        DataCopyPadExtParams<float> padParams;
        padParams.isPad = false;
        padParams.leftPadding = 0;
        padParams.rightPadding = 0;
        padParams.paddingValue = 0.0f;
        DataCopyPad(
            pertokenLocal, pertokenGlobal_[mOffset], params, padParams);
        pertokenQueue_.EnQue(pertokenLocal);
        pertokenLocal = pertokenQueue_.DeQue<float>();
        pertokenLocal.SetSize(tiling_.baseM);
        return pertokenLocal;
    }

    __aicore__ inline void CopyOut(
        const LocalTensor<bfloat16_t> &outLocal, uint32_t currentM,
        uint32_t currentN, uint32_t mOffset, uint32_t nOffset)
    {
        DataCopyExtParams params;
        params.blockCount = currentM;
        params.blockLen = currentN * sizeof(bfloat16_t);
        params.srcStride =
            (tiling_.baseN - currentN) * sizeof(bfloat16_t) / 32U;
        params.dstStride =
            (tiling_.N - currentN) * sizeof(bfloat16_t);
        const uint64_t outputOffset =
            static_cast<uint64_t>(mOffset) * tiling_.N + nOffset;
        DataCopyPad(yGlobal_[outputOffset], outLocal, params);
    }

    __aicore__ inline void ProcessTile(
        uint32_t currentM, uint32_t currentN,
        uint32_t mOffset, uint32_t nOffset)
    {
        LocalTensor<int32_t> mmLocal =
            mmOutQueue_.AllocTensor<int32_t>();
        DataCopyExtParams mmCopyParams;
        mmCopyParams.blockCount = currentM;
        mmCopyParams.blockLen = currentN * sizeof(int32_t);
        mmCopyParams.srcStride =
            (tiling_.N - currentN) * sizeof(int32_t);
        mmCopyParams.dstStride =
            (tiling_.baseN - currentN) * sizeof(int32_t) / 32U;
        DataCopyPadExtParams<int32_t> mmPadParams;
        mmPadParams.isPad = false;
        mmPadParams.leftPadding = 0;
        mmPadParams.rightPadding = 0;
        mmPadParams.paddingValue = 0;
        const uint64_t mmOffset =
            static_cast<uint64_t>(mOffset) * tiling_.N + nOffset;
        DataCopyPad(
            mmLocal, mmWorkspaceGlobal_[mmOffset],
            mmCopyParams, mmPadParams);
        mmOutQueue_.EnQue(mmLocal);
        mmLocal = mmOutQueue_.DeQue<int32_t>();

        LocalTensor<float> scaleLocal = CopyScale(currentN, nOffset);
        LocalTensor<float> pertokenLocal =
            CopyPertoken(currentM, mOffset);

        AscendDequant(
            dequantResult_, mmLocal, scaleLocal, sharedTmp_,
            {static_cast<uint32_t>(currentM),
             static_cast<uint32_t>(tiling_.baseN),
             static_cast<uint32_t>(currentN)});
        PipeBarrier<PIPE_V>();

        const uint32_t dstShape[2] = {
            static_cast<uint32_t>(currentM),
            static_cast<uint32_t>(tiling_.baseN)};
        const uint32_t srcShape[2] = {
            static_cast<uint32_t>(currentM), 1U};
        BroadCast<float, 2, 1>(
            pertokenBroadcast_, pertokenLocal,
            dstShape, srcShape, sharedTmp_);
        PipeBarrier<PIPE_V>();

        const uint32_t computeSize = currentM * tiling_.baseN;
        Mul(
            mulResult_, dequantResult_,
            pertokenBroadcast_, computeSize);
        PipeBarrier<PIPE_V>();

        LocalTensor<bfloat16_t> outLocal =
            outQueue_.AllocTensor<bfloat16_t>();
        Cast(
            outLocal, mulResult_,
            RoundMode::CAST_RINT, computeSize);
        PipeBarrier<PIPE_V>();
        outQueue_.EnQue(outLocal);
        outLocal = outQueue_.DeQue<bfloat16_t>();
        CopyOut(
            outLocal, currentM, currentN, mOffset, nOffset);

        outQueue_.FreeTensor(outLocal);
        pertokenQueue_.FreeTensor(pertokenLocal);
        scaleQueue_.FreeTensor(scaleLocal);
        mmOutQueue_.FreeTensor(mmLocal);
    }

    GlobalTensor<InputType> x1Global_;
    GlobalTensor<InputType> x2Global_;
    GlobalTensor<float> scaleGlobal_;
    GlobalTensor<float> pertokenGlobal_;
    GlobalTensor<bfloat16_t> yGlobal_;
    GlobalTensor<int32_t> mmWorkspaceGlobal_;

    TQue<TPosition::VECIN, 1> mmOutQueue_;
    TQue<TPosition::VECIN, 1> scaleQueue_;
    TQue<TPosition::VECIN, 1> pertokenQueue_;
    TQue<TPosition::VECOUT, 1> outQueue_;
    TBuf<TPosition::VECCALC> workBuffer_;

    LocalTensor<float> dequantResult_;
    LocalTensor<float> pertokenBroadcast_;
    LocalTensor<float> mulResult_;
    LocalTensor<uint8_t> sharedTmp_;

    TCubeTiling tiling_;
    uint32_t tileElements_ = 0;
    uint32_t coreIndex_ = 0;
    uint32_t mIndex_ = 0;
    uint32_t nIndex_ = 0;
    uint32_t mUse_ = 0;
    uint32_t nUse_ = 0;
    bool active_ = false;
};


__global__ __mix__(1, 2) void qmm_custom_kernel(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale,
    GM_ADDR y, __kfc_workspace__ GM_ADDR workspace,
    QmmCustomTilingData tilingData)
{
    TPipe pipe;
    if (tilingData.isPertoken != 0U) {
        QmmPertokenKernel<int8_t> op;
        if ASCEND_IS_AIC {
            REGIST_MATMUL_OBJ(
                &pipe, GetSysWorkSpacePtr(), op.matmulObj,
                &tilingData.cubeTilingData);
        }
        op.Init(
            x1, x2, scale, pertokenScale, y,
            GetUserWorkspace(workspace),
            tilingData.cubeTilingData, &pipe);
        op.Process();
    } else {
        QmmCubeBasicKernel<int8_t> op;
        if ASCEND_IS_AIC {
            REGIST_MATMUL_OBJ(
                &pipe, GetSysWorkSpacePtr(), op.matmulObj,
                &tilingData.cubeTilingData);
        }
        op.Init(x1, x2, y, tilingData.cubeTilingData);
        op.Process();
    }
}

namespace ascendc_ops {

at::Tensor qmm_custom(
    const at::Tensor &x1, const at::Tensor &x2,
    const at::Tensor &scale,
    const c10::optional<at::Tensor> &pertoken_scale)
{
    TORCH_CHECK(x1.dim() == 2 && x2.dim() == 2,
                "qmm_custom expects x1 and x2 to be 2-D tensors");
    TORCH_CHECK(x1.scalar_type() == at::ScalarType::Char &&
                    x2.scalar_type() == at::ScalarType::Char,
                "qmm_custom expects x1/x2 dtype torch.int8");
    TORCH_CHECK(scale.scalar_type() == at::ScalarType::Float,
                "qmm_custom expects scale dtype torch.float32");
    TORCH_CHECK(x1.device() == x2.device() &&
                    x1.device() == scale.device(),
                "x1, x2 and scale must be on the same NPU device");

    const int64_t M = x1.size(0);
    const int64_t K = x1.size(1);
    TORCH_CHECK(x2.size(0) == K,
                "qmm_custom shape mismatch: x1 is [", M, ", ", K,
                "] but x2 is [", x2.size(0), ", ", x2.size(1), "]");
    const int64_t N = x2.size(1);
    TORCH_CHECK(scale.numel() == N,
                "perChannelScale must contain N elements; expected ", N,
                ", got ", scale.numel());

    const uint32_t isPertoken =
        pertoken_scale.has_value() && pertoken_scale.value().defined()
            ? 1U
            : 0U;
    if (isPertoken != 0U) {
        const at::Tensor &tokenScale = pertoken_scale.value();
        TORCH_CHECK(tokenScale.scalar_type() == at::ScalarType::Float,
                    "perTokenScale must have dtype torch.float32");
        TORCH_CHECK(tokenScale.device() == x1.device(),
                    "perTokenScale must be on the same NPU device as x1");
        TORCH_CHECK(tokenScale.numel() == M,
                    "perTokenScale must contain M elements; expected ", M,
                    ", got ", tokenScale.numel());
    }

    const at::ScalarType outputDtype =
        isPertoken != 0U
            ? at::ScalarType::BFloat16
            : at::ScalarType::Int;
    at::Tensor y = at::empty({M, N}, x1.options().dtype(outputDtype));

    QmmCustomTilingData tilingData =
        CalcQmmTiling(isPertoken, M, N, K);
    const uint32_t numBlocks =
        tilingData.cubeTilingData.usedCoreNum;
    TORCH_CHECK(
        tilingData.workspaceSize <=
            static_cast<uint64_t>(std::numeric_limits<int64_t>::max()),
        "workspace is too large for a PyTorch tensor");

    at::Tensor workspaceTensor = at::empty(
        {static_cast<int64_t>(tilingData.workspaceSize)},
        x1.options().dtype(at::ScalarType::Byte));

    auto aclStream = c10_npu::getCurrentNPUStream().stream(true);
    auto *x1Ptr = reinterpret_cast<uint8_t *>(x1.mutable_data_ptr());
    auto *x2Ptr = reinterpret_cast<uint8_t *>(x2.mutable_data_ptr());
    auto *scalePtr = reinterpret_cast<uint8_t *>(scale.mutable_data_ptr());
    uint8_t *pertokenPtr = nullptr;
    if (isPertoken != 0U) {
        pertokenPtr = reinterpret_cast<uint8_t *>(
            pertoken_scale.value().mutable_data_ptr());
    }

    qmm_custom_kernel<<<numBlocks, 0, aclStream>>>(
        x1Ptr, x2Ptr, scalePtr, pertokenPtr,
        reinterpret_cast<uint8_t *>(y.data_ptr()),
        workspaceTensor.data_ptr<uint8_t>(), tilingData);
    return y;
}

}  // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m)
{
    m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, "
          "Tensor? pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m)
{
    m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Writing /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [8]:
!cd {QMM_CUSTOM_DIR} && mkdir -p build && cd build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

-- System processer: aarch64
-- CMAKE_ASC_COMPILER: /home/developer/Ascend/cann-9.0.0/bin/bisheng
-- ASCEND_CANN_PACKAGE_LINUX_PATH: /home/developer/Ascend/cann-9.0.0/aarch64-linux
-- CMAKE_ASC_LLD_LINKER: /home/developer/Ascend/cann-9.0.0/aarch64-linux/ccec_compiler/bin/ld.lld
-- The CXX compiler identification is GNU 9.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /opt/buildtools/Python-3.11.4/bin/python3 (found version "3.11.4") found components: Interpreter Development Development.Module Development.Embed 
CMake Warning at /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchC

编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [9]:
import torch
import torch_npu
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT3

### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [25]:
import os
import csv
from pathlib import Path


# ============================================================
# Profiler 配置
# ============================================================

PROF_OUTPUT_DIR = str(
    SRC_DIR / "prof_output"
)

# warmup 轮数；0 表示不 warmup
WARMUP_ITERS = 0


# ============================================================
# CSV 工具函数
# ============================================================

def parse_float(value):
    """
    安全解析浮点数。

    解析失败时返回 None。
    """
    if value is None:
        return None

    try:
        return float(
            str(value).strip().replace(",", "")
        )
    except (TypeError, ValueError):
        return None


def find_kernel_details_csv(prof_dir):
    """
    在 Profiler 输出目录中递归查找最新的
    kernel_details.csv。
    """
    prof_path = Path(prof_dir)

    if not prof_path.exists():
        return None

    candidates = [
        path
        for path in prof_path.rglob(
            "kernel_details.csv"
        )
        if path.is_file()
    ]

    if not candidates:
        return None

    # 根据修改时间选择最新文件
    latest_csv = max(
        candidates,
        key=lambda path: path.stat().st_mtime,
    )

    return str(latest_csv)


def extract_qmm_kernel_records(csv_path):
    """
    从 kernel_details.csv 中提取 QmmCustom Kernel。

    当前 Profiler 中自定义 Kernel 行的：
        Input Shapes = N/A
        Output Data Types = N/A

    因此这里只读取：
        - Name
        - Type
        - Start Time(us)
        - Duration(us)
        - Task ID

    最后按照实际启动时间排序。
    """
    records = []
    skipped_count = 0

    with open(
        csv_path,
        newline="",
        encoding="utf-8",
        errors="replace",
    ) as csv_file:
        reader = csv.DictReader(csv_file)

        for csv_line, row in enumerate(
            reader,
            start=2,
        ):
            name = str(
                row.get("Name", "") or ""
            )

            op_type = str(
                row.get("Type", "") or ""
            )

            # 同时检查 Name 和 Type
            identity = (
                name + " " + op_type
            ).lower()

            # 兼容以下名称：
            #
            # QmmCustom
            # qmm_custom
            # _Z17qmm_custom_kernel...
            if (
                "qmm_custom_kernel" not in identity
                and "qmmcustom" not in identity
                and "qmm_custom" not in identity
            ):
                continue

            duration_us = parse_float(
                row.get("Duration(us)")
            )

            start_time_us = parse_float(
                row.get("Start Time(us)")
            )

            if duration_us is None:
                skipped_count += 1
                continue

            records.append({
                "csv_line": csv_line,
                "name": name,
                "type": op_type,
                "task_id": row.get("Task ID"),
                "start_time_us": (
                    start_time_us
                    if start_time_us is not None
                    else float("inf")
                ),
                "duration_us": duration_us,
            })

    # 按实际执行时间排序
    records.sort(
        key=lambda record: (
            record["start_time_us"],
            record["csv_line"],
        )
    )

    if skipped_count > 0:
        print(
            f"提示：跳过了 {skipped_count} 条 "
            "Duration 无法解析的 QmmCustom 记录。"
        )

    return records


# ============================================================
# 主流程：Warmup
# ============================================================

int32_results = []
bf16_results = []

if WARMUP_ITERS > 0:
    print("=" * 60)
    print(
        f"Warmup: 运行 {WARMUP_ITERS} "
        "轮，不采集数据"
    )
    print("=" * 60)

    for warmup_round in range(
        WARMUP_ITERS
    ):
        print(
            f"Warmup 轮次："
            f"{warmup_round + 1}/"
            f"{WARMUP_ITERS}"
        )

        for M, K, N in TEST_SHAPES:
            test_int32(M, K, N)
            test_bf16(M, K, N)


# ============================================================
# 创建 Profiler
# ============================================================

os.makedirs(
    PROF_OUTPUT_DIR,
    exist_ok=True,
)

profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=(
        torch_npu.profiler._ExperimentalConfig(
            profiler_level=(
                torch_npu.profiler
                .ProfilerLevel
                .Level1
            ),
            aic_metrics=(
                torch_npu.profiler
                .AiCMetrics
                .PipeUtilization
            ),
        )
    ),
    on_trace_ready=(
        torch_npu.profiler
        .tensorboard_trace_handler(
            PROF_OUTPUT_DIR
        )
    ),
)


# ============================================================
# 开始采集并执行正确性测试
# ============================================================

profiler.start()

for index, (M, K, N) in enumerate(
    TEST_SHAPES,
    start=1,
):
    print("=" * 60)
    print(
        f"规格{index}: "
        f"M={M}, K={K}, N={N}"
    )
    print("=" * 60)

    print("[INT32 测试]")

    int32_pass = test_int32(
        M,
        K,
        N,
    )

    int32_results.append(
        (
            M,
            K,
            N,
            int32_pass,
        )
    )

    profiler.step()

    print("[BF16 测试]")

    bf16_pass = test_bf16(
        M,
        K,
        N,
    )

    bf16_results.append(
        (
            M,
            K,
            N,
            bf16_pass,
        )
    )

    profiler.step()


# ============================================================
# 停止 Profiler
# ============================================================

profiler.stop()


# ============================================================
# 正确性测试汇总
# ============================================================

print()
print("=" * 60)
print("测试汇总")
print("=" * 60)

print(
    f"{'规格':>24} | "
    f"{'INT32':^8} | "
    f"{'BF16':^8}"
)

print("-" * 46)

for (
    M,
    K,
    N,
    int32_pass,
), (
    _,
    _,
    _,
    bf16_pass,
) in zip(
    int32_results,
    bf16_results,
):
    shape_text = (
        f"M={M},K={K},N={N}"
    )

    int32_status = (
        "PASS"
        if int32_pass
        else "FAIL"
    )

    bf16_status = (
        "PASS"
        if bf16_pass
        else "FAIL"
    )

    print(
        f"{shape_text:>24} | "
        f"{int32_status:^8} | "
        f"{bf16_status:^8}"
    )

int32_total = sum(
    bool(result)
    for _, _, _, result
    in int32_results
)

bf16_total = sum(
    bool(result)
    for _, _, _, result
    in bf16_results
)

total = len(TEST_SHAPES)

print(
    f"INT32: {int32_total}/{total} 通过, "
    f"BF16: {bf16_total}/{total} 通过"
)


# ============================================================
# Profiler 数据汇总
# ============================================================

print()
print("=" * 70)
print(
    "Profiler 数据汇总 "
    "(QmmCustom Duration)"
)
print("=" * 70)

csv_path = find_kernel_details_csv(
    PROF_OUTPUT_DIR
)

print(f"原始数据路径：{csv_path}")

if csv_path is None:
    print(
        f"未在 {PROF_OUTPUT_DIR} 中找到 "
        "kernel_details.csv 文件"
    )

    print(
        "请检查 Profiler 输出目录。"
    )

else:
    # 提取所有 QmmCustom Kernel 记录
    qmm_records = (
        extract_qmm_kernel_records(
            csv_path
        )
    )

    actual_record_count = len(
        qmm_records
    )

    # 每个测试规格预期：
    #   1 条 INT32 Kernel
    #   1 条 BF16 Kernel
    expected_record_count = (
        len(TEST_SHAPES) * 2
    )

    if (
        actual_record_count
        < expected_record_count
    ):
        print(
            f"错误：只找到 "
            f"{actual_record_count} 条 "
            "QmmCustom 记录，"
            f"本轮测试预期 "
            f"{expected_record_count} 条。"
        )

        print(
            "Profiler 数据可能未完整生成，"
            "无法输出完整性能表。"
        )

    else:
        if (
            actual_record_count
            > expected_record_count
        ):
            print(
                f"提示：找到 "
                f"{actual_record_count} 条 "
                "QmmCustom 记录，"
                f"本轮测试预期 "
                f"{expected_record_count} 条；"
                f"按时间取最后 "
                f"{expected_record_count} 条。"
            )

            selected_records = (
                qmm_records[
                    -expected_record_count:
                ]
            )

        else:
            print(
                f"提示：找到 "
                f"{actual_record_count} 条 "
                "QmmCustom 记录，"
                "与本轮测试预期数量一致。"
            )

            selected_records = (
                qmm_records
            )

        print(
            "说明：当前 Profiler 的 "
            "QmmCustom 记录未完整携带 "
            "Shape/Dtype，"
            "已依据测试循环顺序完成回填。"
        )

        # --------------------------------------------
        # 按固定执行顺序回填：
        #
        # TEST_SHAPES[0] INT32
        # TEST_SHAPES[0] BF16
        # TEST_SHAPES[1] INT32
        # TEST_SHAPES[1] BF16
        # ...
        # --------------------------------------------

        table_rows = []

        for shape_index, (
            M,
            K,
            N,
        ) in enumerate(TEST_SHAPES):
            int32_record_index = (
                shape_index * 2
            )

            bf16_record_index = (
                int32_record_index + 1
            )

            int32_record = (
                selected_records[
                    int32_record_index
                ]
            )

            bf16_record = (
                selected_records[
                    bf16_record_index
                ]
            )

            table_rows.append({
                "M": M,
                "K": K,
                "N": N,
                "int32_duration_us": (
                    int32_record[
                        "duration_us"
                    ]
                ),
                "bf16_duration_us": (
                    bf16_record[
                        "duration_us"
                    ]
                ),
            })

        # --------------------------------------------
        # 按要求的格式输出最终性能表
        # --------------------------------------------

        header = (
            f"{'规格 (M,K,N)':>24} | "
            f"{'INT32 Duration(us)':>18} | "
            f"{'BF16 Duration(us)':>18}"
        )

        print(header)
        print("-" * len(header))

        for row in table_rows:
            shape_text = (
                f"M={row['M']},"
                f"K={row['K']},"
                f"N={row['N']}"
            )

            int32_text = (
                f"{row['int32_duration_us']:.3f}"
            )

            bf16_text = (
                f"{row['bf16_duration_us']:.3f}"
            )

            print(
                f"{shape_text:>24} | "
                f"{int32_text:>18} | "
                f"{bf16_text:>18}"
            )

print("=" * 70)

规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT3

---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [11]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

93:        x = torch_npu.npu_quant_matmul(x, layer.weight,


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [12]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom

# 1: 导入libascendc_ops.so
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

ct_code = W8A8_FILE.read_text(encoding='utf-8')
_lib_path = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
_load_lib_stmt = 'torch.ops.load_library(r"' + _lib_path + '")'
if _load_lib_stmt not in ct_code:
    # 在 'import torch' 行后插入 'torch.ops.load_library'
    ct_code = ct_code.replace('import torch\n', 'import torch\n' + _load_lib_stmt + '\n')
    W8A8_FILE.write_text(ct_code, encoding='utf-8')
    print('[插入] compressed_tensors.py: 已添加 torch.ops.load_library')
else:
    print('[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过')

# 2: 在 compressed_tensors_w8a8_int8.py 中替换 npu_quant_matmul → qmm_custom
code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.ascendc_ops.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


[替换] 目标文件: /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[插入] compressed_tensors.py: 已添加 torch.ops.load_library
[替换] 成功: npu_quant_matmul → qmm_custom
[验证] 替换已生效，当前使用 qmm_custom 算子


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [17]:
import sys, subprocess, os
from inference_scripts.recipe_workflow import prepare_runtime_yaml

# 将W8A8_WEIGHT设置为量化模型权重实际所在路径
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_A8W8_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM  = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1
[INFO][2026-07-24 00:46:43.140][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-24 00:46:43.140][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-24 00:46:43.141][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-24 00:46:43.145][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-24 00:46:43.145][patch_getenv.py:15] get env RES_PATH = res/20260724/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-07-24 00:46:43.146][infer.py:146] Inference Configuration
[INFO][2026-07-24 00:46:43.146][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitC

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [18]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 256, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [19]:
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1
[INFO][2026-07-24 00:48:49.742][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-24 00:48:49.743][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-24 00:48:49.743][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-24 00:48:49.748][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-24 00:48:49.748][patch_getenv.py:15] get env RES_PATH = res/20260724/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-07-24 00:48:49.748][infer.py:146] Inference Configuration
[INFO][2026-07-24 00:48:49.748][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [20]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'].str.contains('QmmCustom')].copy()
decode_qmm = decode_df[decode_df['Type'].str.contains('QmmCustom')].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

最新结果目录: /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260724
Prefill CSV: /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260724/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/f040d98d16ef4f6b807b282f792dbf96_15567_20260724004935424_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV: /mnt/workspace/gitCode/2301_79608650/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260724/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/f040d98d16ef4f6b807b282f792dbf96_15567_20260724004941658_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Calls


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)